# WORK9 — Clean Feature Selection

Workspace: `/content/drive/MyDrive/work9`. Current lineage is resolved only from fresh Work9 pointers/manifests. After PASS, the pipeline continues to Pair modeling; no old Work8 selection/model artifact is reused.


# WORK9 — Stage 6 Feature Selection V0.4 — Calendar / Lunar lineage

This notebook reads **only** the locked Feature Stage V0.1.3 outputs.

It re-runs Pair/Branch feature selection after adding Gregorian seasonality, working-day/public-holiday proxies, Tết-distance, and Vietnam lunar-month target-calendar features. All calendar features are evaluated under the same locked selection rules as all other candidates; none are force-selected.

No Supabase connection is used. Frozen Test targets 2026-04/05/06 are forbidden. No forecasting model candidate is persisted.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, shutil, subprocess
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import yaml

ROOT = Path('/content/drive/MyDrive/work9')
SRC = ROOT / '02_src' / 'feature_selection'
TEST = ROOT / '07_tests' / 'test_feature_selector_v04.py'
GATE = ROOT / '01_config' / 'work9_stage_gate_v01.yaml'
CONTRACT = ROOT / '01_config' / 'feature_selection_contract_v04.yaml'
FEATURE_POINTER = ROOT / '01_config' / 'current_feature_run.json'

for p in [SRC/'feature_selector_v04.py', TEST, GATE, CONTRACT, FEATURE_POINTER]:
    if not p.exists():
        raise FileNotFoundError(p)

gate = yaml.safe_load(GATE.read_text(encoding='utf-8'))
contract = yaml.safe_load(CONTRACT.read_text(encoding='utf-8'))
feature_pointer = json.loads(FEATURE_POINTER.read_text(encoding='utf-8'))
if not gate['authorization']['feature_selection']:
    raise RuntimeError('WORK9 FEATURE SELECTION BLOCKED')
if gate['authorization']['frozen_test']:
    raise RuntimeError('Unsafe gate: frozen/test must remain false')
if feature_pointer.get('status') != 'PASS':
    raise RuntimeError('Current feature pointer is not PASS')
PAIR_PATH = Path(feature_pointer['pair_feature_panel_path'])
BRANCH_PATH = Path(feature_pointer['branch_feature_panel_path'])
INVENTORY_PATH = Path(feature_pointer['feature_inventory_path'])
FEATURE_MANIFEST = Path(feature_pointer['feature_manifest_path'])
for p in [PAIR_PATH, BRANCH_PATH, INVENTORY_PATH, FEATURE_MANIFEST]:
    if not p.exists(): raise FileNotFoundError(p)
feature_manifest=json.loads(FEATURE_MANIFEST.read_text(encoding='utf-8'))
if feature_manifest.get('status')!='PASS' or feature_manifest.get('run_id')!=feature_pointer['run_id']:
    raise RuntimeError('Current feature manifest mismatch')
if feature_manifest.get('pair_feature_version') != feature_pointer.get('pair_feature_version'):
    raise RuntimeError('Pair feature version mismatch between pointer and manifest')
if feature_manifest.get('branch_feature_version') != feature_pointer.get('branch_feature_version'):
    raise RuntimeError('Branch feature version mismatch between pointer and manifest')
expected_paths = {
    'pair_feature_panel': str(PAIR_PATH),
    'branch_feature_panel': str(BRANCH_PATH),
    'feature_inventory': str(INVENTORY_PATH),
}
for key, expected_path in expected_paths.items():
    if feature_manifest.get('outputs', {}).get(key) != expected_path:
        raise RuntimeError(f'Feature manifest output path mismatch for {key}')
print('Current Work9 feature run verified:', feature_pointer['run_id'])
def json_default(obj):
    """Serialize numpy/pandas scalar types without changing semantic types."""
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


Current Work9 feature run verified: feature_stage_v013_20260815T123431Z


In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

expected_hashes = feature_pointer.get('output_sha256')
if not expected_hashes:
    raise RuntimeError('current_feature_run.json lacks output_sha256; rerun patched Feature Build notebook 02')

pair_hash = sha256_file(PAIR_PATH)
branch_hash = sha256_file(BRANCH_PATH)
inventory_hash = sha256_file(INVENTORY_PATH)
actual_hashes = {
    'pair_feature_panel': pair_hash,
    'branch_feature_panel': branch_hash,
    'feature_inventory': inventory_hash,
}
for key, actual in actual_hashes.items():
    expected = expected_hashes.get(key)
    if actual != expected:
        raise RuntimeError(f'{key} SHA256 mismatch: expected={expected}, actual={actual}')
print('Current Work9 feature input SHA256 verified against current_feature_run.json.')


Current Work9 feature input SHA256 verified against current_feature_run.json.


In [ ]:
# Run Stage-6 unit tests before touching the real feature panels.
# pytest runs in a fresh subprocess, so explicitly pass the Stage-6 source path.
test_env = os.environ.copy()
existing_pythonpath = test_env.get('PYTHONPATH', '')
test_env['PYTHONPATH'] = str(SRC) + (os.pathsep + existing_pythonpath if existing_pythonpath else '')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', str(TEST)],
    text=True, capture_output=True, env=test_env,
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError('Stage-6 tests failed')
print('Stage-6 unit tests PASS.')


.........                                                                [100%]
9 passed in 3.69s

Stage-6 unit tests PASS.


In [ ]:
sys.path.insert(0, str(SRC))
from feature_selector_v04 import (
    SelectionConfig, select_track, validate_selection_result, write_yaml_feature_list,
    PAIR_FAMILIES, BRANCH_FAMILIES, PAIR_TARGET, BRANCH_TARGET, sha256_file as code_sha_file,
    SELECTION_VERSION,
)

pair_df = pd.read_parquet(PAIR_PATH)
branch_df = pd.read_parquet(BRANCH_PATH)
inventory = pd.read_csv(INVENTORY_PATH)

# Hard boundary: Stage-6 inputs themselves may not contain Frozen-Test targets.
if pd.to_datetime(pair_df['target_month']).ge(pd.Timestamp(contract['data_scope']['frozen_test_start'])).any():
    raise RuntimeError('Pair input contains Frozen-Test target rows')
if pd.to_datetime(branch_df['target_month']).ge(pd.Timestamp(contract['data_scope']['frozen_test_start'])).any():
    raise RuntimeError('Branch input contains Frozen-Test target rows')

sc = contract['screening_estimator']
imp = contract['importance']
sel = contract['selection']
static = contract['static_screen']
cfg = SelectionConfig(
    missing_rate_drop=float(static['missing_rate_drop']),
    corr_review_threshold=float(static['correlation_review_threshold']),
    permutation_repeats=int(imp['repeats']),
    min_perm_wape_gain=float(imp['minimum_wape_gain']),
    pair_min_selected=int(sel['pair_min_features']),
    branch_min_selected=int(sel['branch_min_features']),
    random_state=int(sc['random_state']),
    pair_max_train_rows=int(sc['pair_max_train_rows']),
    branch_max_train_rows=int(sc['branch_max_train_rows']),
    screening_max_iter=int(sc['max_iter']),
    screening_learning_rate=float(sc['learning_rate']),
    screening_max_leaf_nodes=int(sc['max_leaf_nodes']),
    screening_l2=float(sc['l2_regularization']),
)
print('Loaded:', pair_df.shape, branch_df.shape, inventory.shape)
print('PRIMARY validation = CURRENT ACTIVE portfolio; SECONDARY = ALL diagnostic')


Loaded: (1186159, 121) (3468, 83) (202, 8)
PRIMARY validation = CURRENT ACTIVE portfolio; SECONDARY = ALL diagnostic


In [ ]:
pair_result = select_track(
    pair_df, inventory, 'PAIR', PAIR_TARGET, PAIR_FAMILIES, cfg,
    max_train_rows=cfg.pair_max_train_rows, min_selected=cfg.pair_min_selected,
)
print('PAIR screening WAPE:', pair_result['full_metrics'])
print('PAIR selected:', len(pair_result['selected']))

print('PAIR secondary ALL validation WAPE:', pair_result['secondary_all_validation_metrics'])
print('PAIR primary/secondary rows:', pair_result['validation_rows_primary_current_active'], pair_result['validation_rows_secondary_all'])


PAIR screening WAPE: {'wape': 1.1250538847254414, 'h1_wape': 0.8816784940287405, 'h2_wape': 2.236320795453793, 'h3_wape': 1.0832535833425037}
PAIR selected: 54
PAIR secondary ALL validation WAPE: {'wape': 1.1626210347705441, 'h1_wape': 0.9110589187945889, 'h2_wape': 2.2953011956502016, 'h3_wape': 1.1260197261672054}
PAIR primary/secondary rows: 44289 77886


In [ ]:
branch_result = select_track(
    branch_df, inventory, 'BRANCH', BRANCH_TARGET, BRANCH_FAMILIES, cfg,
    max_train_rows=cfg.branch_max_train_rows, min_selected=cfg.branch_min_selected,
)
print('BRANCH screening WAPE:', branch_result['full_metrics'])
print('BRANCH selected:', len(branch_result['selected']))

print('BRANCH secondary ALL validation WAPE:', branch_result['secondary_all_validation_metrics'])
print('BRANCH primary/secondary rows:', branch_result['validation_rows_primary_current_active'], branch_result['validation_rows_secondary_all'])


BRANCH screening WAPE: {'wape': 0.46975383913253344, 'h1_wape': 0.32761489481813755, 'h2_wape': 1.3878479921940456, 'h3_wape': 0.388973031649852}
BRANCH selected: 35
BRANCH secondary ALL validation WAPE: {'wape': 0.46975383913253344, 'h1_wape': 0.3276148948181377, 'h2_wape': 1.387847992194046, 'h3_wape': 0.38897303164985186}
BRANCH primary/secondary rows: 171 180


In [ ]:
pair_validation = validate_selection_result(pair_result, 'PAIR', cfg.pair_min_selected)
branch_validation = validate_selection_result(branch_result, 'BRANCH', cfg.branch_min_selected)
validation = {
    'status': 'PASS' if pair_validation['status']=='PASS' and branch_validation['status']=='PASS' else 'FAIL',
    'pair': pair_validation,
    'branch': branch_validation,
    'frozen_test_touched': False,
    'model_candidate_persisted': False,
}
print(json.dumps(validation, indent=2, default=json_default))
if validation['status'] != 'PASS':
    raise RuntimeError('Stage-6 selection validation failed')


{
  "status": "PASS",
  "pair": {
    "track": "PAIR",
    "status": "PASS",
    "checks": {
      "nonempty_selected_list": true,
      "selected_subset_static_kept": true,
      "no_constant_selected": true,
      "no_high_missing_selected": true,
      "no_exact_duplicate_selected": true,
      "horizon_present_for_pooled": true,
      "validation_wape_finite": true,
      "primary_validation_nonempty": true,
      "secondary_validation_not_smaller": true,
      "calendar_v013_candidates_seen": true,
      "no_target_or_snapshot_selected": true
    }
  },
  "branch": {
    "track": "BRANCH",
    "status": "PASS",
    "checks": {
      "nonempty_selected_list": true,
      "selected_subset_static_kept": true,
      "no_constant_selected": true,
      "no_high_missing_selected": true,
      "no_exact_duplicate_selected": true,
      "horizon_present_for_pooled": true,
      "validation_wape_finite": true,
      "primary_validation_nonempty": true,
      "secondary_validation_not_small

In [10]:
RUN_ID = 'feature_selection_v04_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = ROOT / '08_runs' / RUN_ID
REPORT_DIR = ROOT / '06_reports' / 'feature_selection' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
REPORT_DIR.mkdir(parents=True, exist_ok=False)

# Reports
pair_result['quality_report'].to_csv(REPORT_DIR/'pair_feature_quality_report.csv', index=False)
branch_result['quality_report'].to_csv(REPORT_DIR/'branch_feature_quality_report.csv', index=False)
pair_result['permutation_importance'].to_csv(REPORT_DIR/'pair_permutation_importance.csv', index=False)
branch_result['permutation_importance'].to_csv(REPORT_DIR/'branch_permutation_importance.csv', index=False)
pair_result['family_ablation'].to_csv(REPORT_DIR/'pair_family_ablation.csv', index=False)
branch_result['family_ablation'].to_csv(REPORT_DIR/'branch_family_ablation.csv', index=False)
pair_result['correlation_review'].to_csv(REPORT_DIR/'pair_correlation_review.csv', index=False)
branch_result['correlation_review'].to_csv(REPORT_DIR/'branch_correlation_review.csv', index=False)

write_yaml_feature_list(REPORT_DIR/'pair_selected_feature_list.yaml', 'PAIR', pair_result['selected'], feature_pointer['run_id'], RUN_ID)
write_yaml_feature_list(REPORT_DIR/'branch_selected_feature_list.yaml', 'BRANCH', branch_result['selected'], feature_pointer['run_id'], RUN_ID)
(REPORT_DIR/'feature_selection_validation.json').write_text(json.dumps(validation, indent=2, default=json_default), encoding='utf-8')

# Immutable run snapshot of contracts and result files.
contracts_dir = RUN_DIR/'contracts'
contracts_dir.mkdir(parents=True)
required_snapshot_files = [
    GATE, CONTRACT,
    ROOT/'01_config'/'evaluation_contract_v02.yaml',
    ROOT/'01_config'/'production_universe_contract_v02.yaml',
    ROOT/'00_docs'/'EVALUATION_UNIVERSE_SPEC_V1.1.md',
    ROOT/'00_docs'/'FEATURE_SELECTION_SPEC_V0.4.md',
    FEATURE_POINTER, FEATURE_MANIFEST,
]
for p in required_snapshot_files:
    if not p.exists():
        raise FileNotFoundError(p)
    shutil.copy2(p, contracts_dir/p.name)

# Work9 clean migration intentionally does not depend on legacy Work8 docs.
# Copy current documentation when present, but do not fail a completed
# feature-selection run because an archival prose file was not migrated.
optional_snapshot_docs = [
    ROOT/'00_docs'/'00_CURRENT_INDEX.md',
]
for p in optional_snapshot_docs:
    if p.exists():
        shutil.copy2(p, contracts_dir/p.name)
for p in REPORT_DIR.iterdir():
    if p.is_file():
        shutil.copy2(p, RUN_DIR/p.name)

manifest = {
    'run_id': RUN_ID,
    'run_type': 'FEATURE_SELECTION_V04',
    'status': 'PASS',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_feature_run_id': feature_pointer['run_id'],
    'dataset_version': feature_manifest['dataset_version'],
    'pair_feature_version': feature_manifest['pair_feature_version'],
    'branch_feature_version': feature_manifest['branch_feature_version'],
    'selection_version': SELECTION_VERSION,
    'validation_policy': {'primary': 'CURRENT_ACTIVE_PORTFOLIO', 'secondary': 'ALL_DIAGNOSTIC_ONLY'},
    'input_sha256': {'pair_feature_panel': pair_hash, 'branch_feature_panel': branch_hash, 'feature_inventory': inventory_hash},
    'code_sha256': code_sha_file(SRC/'feature_selector_v04.py'),
    'contract_sha256': code_sha_file(CONTRACT),
    'source_feature_manifest_sha256': code_sha_file(FEATURE_MANIFEST),
    'row_counts': {
        'pair_train_eligible': pair_result['train_rows'],
        'pair_validation_primary_current_active': pair_result['validation_rows_primary_current_active'],
        'pair_validation_secondary_all': pair_result['validation_rows_secondary_all'],
        'branch_train_eligible': branch_result['train_rows'],
        'branch_validation_primary_current_active': branch_result['validation_rows_primary_current_active'],
        'branch_validation_secondary_all': branch_result['validation_rows_secondary_all'],
    },
    'selected_counts': {'pair': len(pair_result['selected']), 'branch': len(branch_result['selected'])},
    'screening_metrics_primary': {'pair': pair_result['full_metrics'], 'branch': branch_result['full_metrics']},
    'screening_metrics_secondary_all': {'pair': pair_result['secondary_all_validation_metrics'], 'branch': branch_result['secondary_all_validation_metrics']},
    'outputs': {
        'report_dir': str(REPORT_DIR),
        'pair_selected_feature_list': str(REPORT_DIR/'pair_selected_feature_list.yaml'),
        'branch_selected_feature_list': str(REPORT_DIR/'branch_selected_feature_list.yaml'),
    },
    'safety': {
        'supabase_accessed': False,
        'frozen_test_touched': False,
        'screening_estimator_persisted': False,
        'model_candidate_created': False,
        'pair_modeling_run': False,
        'branch_modeling_run': False,
        'production_published': False,
    },
}
(RUN_DIR/'feature_selection_manifest.json').write_text(json.dumps(manifest, indent=2, default=json_default), encoding='utf-8')
(REPORT_DIR/'feature_selection_manifest.json').write_text(json.dumps(manifest, indent=2, default=json_default), encoding='utf-8')

# Publish only the latest CURRENT Work9 selected-feature lists.
SELECTED_DIR = ROOT / '05_selected_features'
SELECTED_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(REPORT_DIR/'pair_selected_feature_list.yaml', SELECTED_DIR/'pair_selected_feature_list_v04.yaml')
shutil.copy2(REPORT_DIR/'branch_selected_feature_list.yaml', SELECTED_DIR/'branch_selected_feature_list_v04.yaml')
shutil.copy2(REPORT_DIR/'feature_selection_manifest.json', SELECTED_DIR/'feature_selection_manifest_v04.json')
current_selection = {
    'run_id': RUN_ID, 'status': 'PASS', 'selection_version': SELECTION_VERSION,
    'source_feature_run_id': feature_pointer['run_id'],
    'pair_selected_path': str(SELECTED_DIR/'pair_selected_feature_list_v04.yaml'),
    'branch_selected_path': str(SELECTED_DIR/'branch_selected_feature_list_v04.yaml'),
    'manifest_path': str(SELECTED_DIR/'feature_selection_manifest_v04.json'),
    'validation_primary': 'CURRENT_ACTIVE_PORTFOLIO',
    'validation_secondary': 'ALL_DIAGNOSTIC_ONLY',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
}
(ROOT/'01_config'/'current_feature_selection_run.json').write_text(json.dumps(current_selection, indent=2, default=json_default), encoding='utf-8')

print('PASS — Stage 6 candidate selection run:', RUN_ID)
print('PAIR selected features:', pair_result['selected'])
print('BRANCH selected features:', branch_result['selected'])
print('No Frozen Test was accessed. No model candidate was persisted.')


PASS — Stage 6 candidate selection run: feature_selection_v04_20260815T130048Z
PAIR selected features: ['sku_global_roll_3_mean', 'sku_global_lag_3', 'sku_global_lag_1', 'size_branch_target_available_sku_count_lag_1', 'pair_lag_1', 'pair_roll_3_mean', 'pair_roll_6_mean', 'sku_global_positive_rate_12m', 'sku_known_branch_count_to_origin', 'pair_roll_3_sum', 'pair_roll_12_sum', 'pair_roll_6_sum', 'sku_global_roll_6_mean', 'pair_positive_rate_to_origin', 'target_working_days_proxy', 'pair_cv2_positive_to_origin', 'tile_size_code', 'size_branch_lag_2', 'size_branch_roll_12_mean', 'sku_global_roll_12_mean', 'pair_roll_3_std', 'pair_lag_2', 'size_branch_positive_sku_count_lag_1', 'horizon', 'target_month_sin', 'months_since_last_positive', 'pair_roll_12_mean', 'target_days_since_prev_tet_from_midmonth', 'size_branch_lag_1', 'pair_lag_12', 'target_days_to_next_tet_from_midmonth', 'sku_global_roll_6_sum', 'target_public_holiday_event_count', 'sku_global_lag_12', 'target_lunar_month_mid', 'pair